# 03 - Label Consolidation, Splits & Baselines

**Support Ticket Triage** - Notebook 3 of 6

Notebook 02 settled the dataset question at a matched sample size of 8,469:

| Dataset / target | dummy | TF-IDF | lift | |
|---|---|---|---|---|
| Suraj520 Ticket Type | 0.1865 | 0.2016 | +0.0151 | FAIL |
| Suraj520 Ticket Priority | 0.2431 | 0.2629 | +0.0198 | FAIL |
| Suraj520 Ticket Subject | 0.0638 | 0.0606 | -0.0032 | FAIL |
| CFPB Product | 0.0533 | 0.4877 | **+0.4344** | PASS |
| CFPB Issue (top 15) | 0.0600 | 0.5002 | **+0.4403** | PASS |

This notebook prepares CFPB for training. The order of operations is load-bearing
throughout:

1. **Consolidate the taxonomy** - CFPB renamed its categories in 2017, so old and new
   names coexist as duplicate classes no model can separate from text.
2. **Clean** - CFPB redacts PII as `XXXX`, which would otherwise be a top-frequency token.
3. **Deduplicate, then split** - identical narratives must not straddle train and test.
4. **Split each task separately** - see the note below; this replaces an earlier design.
5. **Baselines** - TF-IDF + Logistic Regression, fit on train, scored on validation.

> **Design change from the first version of this notebook.** Product and Issue originally
> shared one set of splits, which required restricting the corpus to the top 15 issues.
> That deleted 99% of the `Credit card or prepaid card` product class - 41,660 rows down
> to ~360 - because credit-card complaints carry issues ranked 17-24. Three product
> classes ended up with 15-53 validation rows and dragged macro-F1 to 0.6105 while
> accuracy sat at 0.8643. The 25-point gap was entirely starved classes.
>
> Each task now gets its own splits drawn from its own population. Two sealed test sets
> instead of one; both tasks well-posed.

*Accelerator: **None (CPU)**. Internet: **ON** - the dataset is pulled via `kagglehub`.*

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
sns.set_theme(style="whitegrid")

SEED          = 42
SAMPLE_N      = 100_000   # rows per task
TOP_ISSUES    = 15
MIN_PER_CLASS = 200       # applied AFTER subsampling - see note in the split function

WORK   = "/kaggle/working"
SPLITS = os.path.join(WORK, "splits")
PLOTS  = os.path.join(WORK, "plots")
os.makedirs(SPLITS, exist_ok=True)
os.makedirs(PLOTS, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, name + ".png"), dpi=120, bbox_inches="tight")
    plt.show()

def find_files(base, exts=(".csv", ".tsv")):
    out = []
    if not os.path.isdir(base):
        return out
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(exts):
                out.append(os.path.join(root, f))
    return out

print("config ok")

### Load from code

`kagglehub` downloads and caches the dataset, so no *Add Input* step is needed.
Requires **Settings -> Internet ON**.

In [ ]:
import kagglehub

CFPB_SLUG = "selener/consumer-complaint-database"
base = kagglehub.dataset_download(CFPB_SLUG)   # returns a DIRECTORY, not a file
print("dataset at:", base)

cfpb_files = find_files(base)
for p in cfpb_files:
    print(f"  {p}  ({os.path.getsize(p) / 1e6:.0f} MB)")
assert cfpb_files, "No CSV found in the downloaded dataset."

CFPB = max(cfpb_files, key=os.path.getsize)
print("\nusing:", CFPB)

NARR, PROD, ISSU = "Consumer complaint narrative", "Product", "Issue"
cf = pd.read_csv(CFPB, usecols=[NARR, PROD, ISSU], dtype=str, low_memory=False)
cf = cf.dropna(subset=[NARR, PROD, ISSU])
cf = cf[cf[NARR].str.strip().str.len() > 20]
print("rows with a usable narrative:", len(cf))

---
## 1. Label consolidation

CFPB overhauled its taxonomy in 2017 and both vocabularies coexist in the database.
`Credit reporting` (31,554) and
`Credit reporting, credit repair services, or other personal consumer reports` (92,338)
are the same thing under two names. **Nothing in the narrative separates them - only the
filing date does.** Leaving them apart asks the model to guess a timestamp, and suppresses
macro-F1 for reasons that have nothing to do with the model.

In [ ]:
print(f"=== all {cf[PROD].nunique()} raw Product classes ===")
print(cf[PROD].value_counts().to_string())

In [ ]:
# Old taxonomy name -> current name. Only pairs that are the same concept renamed.
PRODUCT_MERGES = {
    "Credit reporting":
        "Credit reporting, credit repair services, or other personal consumer reports",
    "Credit card":             "Credit card or prepaid card",
    "Prepaid card":            "Credit card or prepaid card",
    "Bank account or service": "Checking or savings account",
    "Money transfers":         "Money transfer, virtual currency, or money service",
    "Virtual currency":        "Money transfer, virtual currency, or money service",
    "Payday loan":             "Payday loan, title loan, or personal loan",
}

# Dropped rather than merged - these map to MULTIPLE modern classes, so no single
# assignment is correct:
#   'Consumer Loan'           -> split into vehicle / personal / title loans in 2017
#   'Other financial service' -> catch-all with no consistent meaning
DROP_PRODUCTS = ["Consumer Loan", "Other financial service"]

before = cf[PROD].nunique()
cf[PROD] = cf[PROD].replace(PRODUCT_MERGES)
n_dropped = int(cf[PROD].isin(DROP_PRODUCTS).sum())
cf = cf[~cf[PROD].isin(DROP_PRODUCTS)]

print(f"Product classes: {before} -> {cf[PROD].nunique()}")
print(f"dropped {n_dropped} rows in ambiguous classes {DROP_PRODUCTS}\n")
print(cf[PROD].value_counts().to_string())

In [ ]:
ISSUE_MERGES = {
    "Incorrect information on credit report": "Incorrect information on your report",
    "Cont'd attempts collect debt not owed":  "Attempts to collect debt not owed",
    "Dealing with my lender or servicer":     "Dealing with your lender or servicer",
    "Disclosure verification of debt":        "Written notification about debt",
}

before = cf[ISSU].nunique()
cf[ISSU] = cf[ISSU].replace(ISSUE_MERGES)   # merge BEFORE ranking - merging changes counts
print(f"Issue classes: {before} -> {cf[ISSU].nunique()}")

print("\n=== top 20 Issue classes after merging ===")
print(cf[ISSU].value_counts().head(20).to_string())

---
## 2. Text cleaning

CFPB redacts personal data as runs of `X` - `XX/XX/XXXX` for dates, `XXXX` for names and
account numbers, `{$1,234.00}` for amounts.

**This function is copied verbatim into the Space's `src/preprocessing.py`.** Keep it
dependency-free so train-time and serve-time cleaning cannot drift.

In [ ]:
def clean_text(s):
    """Normalise a CFPB complaint narrative. Pure-Python, no dependencies."""
    s = str(s).lower()
    s = re.sub(r"x{2,}[/\-]x{2,}[/\-]x{2,}", " ", s)   # XX/XX/XXXX redacted dates
    s = re.sub(r"\bx{2,}\b", " ", s)                    # XXXX redacted PII
    s = re.sub(r"\{\$[^}]*\}", " ", s)                  # {$1,234.00} redacted amounts
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

sample = cf[NARR].sample(5000, random_state=SEED)
x_share = (sample.str.lower().str.count(r"\bx{2,}\b").sum()
           / sample.str.split().str.len().sum())
print(f"share of tokens that are XXXX redactions: {x_share:.2%}\n")

print("--- before ---");  print(sample.iloc[0][:320])
print("\n--- after ---"); print(clean_text(sample.iloc[0])[:320])

cf["text"] = cf[NARR].map(clean_text)
cf = cf[cf["text"].str.split().str.len() >= 5]
print(f"\nrows after cleaning: {len(cf)}")

### Deduplicate once, globally

22k narratives are exact duplicates - the same complaint filed against several companies,
or resubmitted. Deduplicating **before** any splitting is what stops identical text
appearing in both train and test.

In [ ]:
before = len(cf)
cf = cf.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print(f"deduplicated: {before} -> {len(cf)} (removed {before - len(cf)})")

cf["id"] = np.arange(len(cf))
cf = cf[["id", "text", PROD, ISSU]].rename(columns={PROD: "product", ISSU: "issue"})
cf.head(3)

### Real token lengths

`max_length` is the biggest single lever on Week 2 training time, so it is measured with
the actual DistilBERT tokenizer rather than a words-to-wordpieces heuristic.

In [ ]:
try:
    from transformers import DistilBertTokenizerFast
    tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    lens = np.array([len(tok.encode(t, truncation=False))
                     for t in cf["text"].sample(3000, random_state=SEED)])
    print(pd.Series(lens).describe().round(1).to_string())
    for m in (128, 256, 384, 512):
        print(f"  truncated at max_length={m}: {(lens > m).mean():.1%}")
except Exception as e:
    print("tokenizer check skipped (needs Internet ON):", e)

---
## 3. Per-task splits

Each task draws from its own population:

| Task | Population | Classes |
|---|---|---|
| `product` | every cleaned row | all consolidated products |
| `issue` | rows whose issue is in the top 15 | 15 |

**The rare-class filter runs after subsampling, not before.** A class holding 300 rows in
a 330k frame lands at ~90 in a 100k sample; filtering on the population would let it
through and then break stratification. This is the same ordering bug that broke the gate
in Notebook 02, so it is worth stating explicitly rather than relying on memory.

In [ ]:
def make_splits(df, label_col, task, sample_n=SAMPLE_N,
                min_per_class=MIN_PER_CLASS, seed=SEED):
    """Subsample -> drop rare classes -> stratified 70/15/15 -> verify -> write."""
    d = df[["id", "text", label_col]].rename(columns={label_col: "label"}).dropna()

    if len(d) > sample_n:
        d = d.sample(sample_n, random_state=seed)

    vc = d["label"].value_counts()
    keep = vc[vc >= min_per_class].index
    n_drop = int((~d["label"].isin(keep)).sum())
    if n_drop:
        print(f"  dropped {n_drop} rows in {len(vc) - len(keep)} classes "
              f"with < {min_per_class} rows after subsampling")
    d = d[d["label"].isin(keep)].reset_index(drop=True)

    tr, tmp = train_test_split(d, test_size=0.30,
                               stratify=d["label"], random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.50,
                               stratify=tmp["label"], random_state=seed)

    a, b, c = set(tr["id"]), set(va["id"]), set(te["id"])
    assert not (a & b or a & c or b & c), "split overlap!"
    assert len(a | b | c) == len(d)

    out = os.path.join(SPLITS, task)
    os.makedirs(out, exist_ok=True)
    for name, part in (("train", tr), ("val", va), ("test", te)):
        part.to_csv(os.path.join(out, f"{name}.csv"), index=False)

    comp = pd.DataFrame({
        "train": tr["label"].value_counts(normalize=True),
        "val":   va["label"].value_counts(normalize=True),
        "test":  te["label"].value_counts(normalize=True),
    })
    comp["dev_pp"] = ((comp.max(axis=1) - comp.min(axis=1)) * 100).round(2)
    worst = comp["dev_pp"].max()

    print(f"[{task}] {len(d)} rows | {d['label'].nunique()} classes | "
          f"train {len(tr)} / val {len(va)} / test {len(te)}")
    print(f"  smallest class: {d['label'].value_counts().min()} rows")
    print(f"  worst split deviation: {worst:.2f} pp",
          "OK" if worst < 1.0 else "<-- CHECK")
    print((comp[["train", "val", "test"]] * 100).round(2)
          .join(comp["dev_pp"]).to_string(), "\n")
    return tr, va, te

In [ ]:
print("=== product: full population ===")
prod_tr, prod_va, prod_te = make_splits(cf, "product", "product")

print("=== issue: top-15 population ===")
top_issues = cf["issue"].value_counts().head(TOP_ISSUES).index.tolist()
issue_pop = cf[cf["issue"].isin(top_issues)]
print(f"  restricted to top {TOP_ISSUES} issues: {len(issue_pop)} of {len(cf)} rows "
      f"({len(issue_pop)/len(cf):.1%})")
issue_tr, issue_va, issue_te = make_splits(issue_pop, "issue", "issue")

### The test-set firewall

Sealed in code, not by discipline. Paste this into Notebooks 04 and 05 unchanged; only
Notebook 06 flips the flag.

In [ ]:
ALLOW_TEST = False   # set True ONLY in notebook 06

def load_split(task, name, base=SPLITS):
    if name == "test" and not ALLOW_TEST:
        raise RuntimeError(f"{task}/test is sealed until notebook 06.")
    return pd.read_csv(os.path.join(base, task, f"{name}.csv"))

for task in ("product", "issue"):
    try:
        load_split(task, "test")
        print(f"FIREWALL BROKEN for {task}")
    except RuntimeError as e:
        print("sealed:", e)

label_maps = {
    "product": {
        "classes": sorted(prod_tr["label"].unique().tolist()),
        "num_labels": int(prod_tr["label"].nunique()),
        "n_rows": int(len(prod_tr) + len(prod_va) + len(prod_te)),
    },
    "issue": {
        "classes": sorted(issue_tr["label"].unique().tolist()),
        "num_labels": int(issue_tr["label"].nunique()),
        "n_rows": int(len(issue_tr) + len(issue_va) + len(issue_te)),
    },
    "product_merges": PRODUCT_MERGES,
    "issue_merges":   ISSUE_MERGES,
    "dropped_products": DROP_PRODUCTS,
    "top_issues": TOP_ISSUES,
    "seed": SEED,
}
with open(os.path.join(WORK, "label_maps.json"), "w") as f:
    json.dump(label_maps, f, indent=2)

print(f"\nnum_labels -> product: {label_maps['product']['num_labels']}, "
      f"issue: {label_maps['issue']['num_labels']}")

---
## 4. Baselines

Fit on **train only**, scored on **validation**. The stratified-dummy row stays in every
results table from here on, so a later "DistilBERT beat the baseline" claim can always be
checked against chance.

In [ ]:
def baseline(task, tr, va):
    dummy = DummyClassifier(strategy="stratified", random_state=SEED).fit(
        tr["text"], tr["label"])
    f1_d = f1_score(va["label"], dummy.predict(va["text"]), average="macro")

    pipe = make_pipeline(
        TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                        min_df=2, sublinear_tf=True, strip_accents="unicode"),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    ).fit(tr["text"], tr["label"])
    pred = pipe.predict(va["text"])
    f1_t = f1_score(va["label"], pred, average="macro")
    acc  = accuracy_score(va["label"], pred)

    print(f"\n########## {task} ##########")
    print(f"dummy  macro-F1 = {f1_d:.4f}")
    print(f"TF-IDF macro-F1 = {f1_t:.4f}   accuracy = {acc:.4f}")
    print(f"gap (acc - macro) = {acc - f1_t:+.4f}  "
          "<- a large gap means some classes are starved")
    print("\n" + classification_report(va["label"], pred, zero_division=0))

    dump(pipe, os.path.join(WORK, f"tfidf_lr_{task}.joblib"))
    return {"task": task, "n_classes": int(tr["label"].nunique()),
            "n_train": int(len(tr)),
            "f1_dummy": round(float(f1_d), 4), "f1_tfidf": round(float(f1_t), 4),
            "accuracy_tfidf": round(float(acc), 4),
            "acc_macro_gap": round(float(acc - f1_t), 4)}

baselines = [baseline("product", prod_tr, prod_va),
             baseline("issue",   issue_tr, issue_va)]

### Did the redesign work?

The shared-splits version scored **0.6105** macro-F1 on product against **0.8643**
accuracy - a 25-point gap created by three classes with 15-53 validation rows. If the
per-task splits fixed it, product macro-F1 should rise and the gap should shrink.

In [ ]:
bl = pd.DataFrame(baselines)
display(bl)

prev = {"f1_tfidf": 0.6105, "accuracy_tfidf": 0.8643, "gap": 0.2538}
now  = bl[bl["task"] == "product"].iloc[0]
print("product, shared splits (old):  macro-F1 0.6105  acc 0.8643  gap +0.2538")
print(f"product, per-task splits (new): macro-F1 {now['f1_tfidf']:.4f}  "
      f"acc {now['accuracy_tfidf']:.4f}  gap {now['acc_macro_gap']:+.4f}")
print(f"\nmacro-F1 change: {now['f1_tfidf'] - prev['f1_tfidf']:+.4f}")

metrics = {
    "splits": {t: {n: int(len(d)) for n, d in
                   zip(("train", "val", "test"), parts)}
               for t, parts in (("product", (prod_tr, prod_va, prod_te)),
                                ("issue",   (issue_tr, issue_va, issue_te)))},
    "baselines_on_validation": baselines,
    "superseded_shared_split_product": prev,
    "note": "DistilBERT must beat f1_tfidf on validation to justify its cost.",
}
with open(os.path.join(WORK, "baseline_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

melt = bl.melt(id_vars="task", value_vars=["f1_dummy", "f1_tfidf"],
               var_name="model", value_name="macro_f1")
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=melt, x="task", y="macro_f1", hue="model")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9)
ax.set(title="Validation baselines, per-task splits (CFPB)", ylim=(0, 1))
savefig("06_baselines")

print("\nartifacts:", sorted(os.listdir(WORK)))

---
## Next

**Save Version -> Save & Run All**, then add this notebook's output as an input to
Notebooks 04 and 05.

Carry forward:

- `splits/product/{train,val,test}.csv` and `splits/issue/{...}` - both test sets sealed
- `label_maps.json` - **`num_labels` comes from here**, never a hardcoded number
- `baseline_metrics.json` - the bar DistilBERT has to clear
- `max_length` from the tokenizer check (256 unless the numbers argue otherwise)
- `clean_text()` - copied verbatim into the Space later

**Notebook 04** fine-tunes DistilBERT on `product`, **05** on `issue`. Both need GPU +
Internet, and both must `pip install "transformers<5" tf-keras` and set
`TF_USE_LEGACY_KERAS=1` **before** importing transformers.